In [15]:
from pymodbus.client import ModbusTcpClient
import time


# ============================================================
# 1. Modbus TCP 설정
# ============================================================

SERVER_IP = "210.119.14.74"
SERVER_PORT = 502
SLAVE_ID = 1

client = ModbusTcpClient(
    SERVER_IP,
    port=SERVER_PORT,
    timeout=3
)


# ============================================================
# 2. Factory I/O Discrete Input 주소
# ============================================================

ITEM_AT_ENTRY = 0
ITEM_AT_EXIT = 1
MOVING_X = 2
MOVING_Z = 3
ITEM_DETECTED = 4

START_BUTTON = 5
RESET_BUTTON = 6
STOP_BUTTON = 7
EMERGENCY_STOP = 8
AUTO_MODE = 9
FACTORY_RUNNING = 10


# ============================================================
# 3. Factory I/O Coil 주소
# ============================================================

ENTRY_CONVEYOR = 0
EXIT_CONVEYOR = 1
MOVE_X = 2
MOVE_Z = 3
GRAB = 4

START_LIGHT = 5
RESET_LIGHT = 6
STOP_LIGHT = 7

# Emitter 0 (Emit)
EMITTER = 8


# ============================================================
# 4. Holding Register 주소
# ============================================================

# 처리한 전체 부품 수
COUNTER_REGISTER = 0

# Emitter 0 (Part)
EMITTER_PART_REGISTER = 1

# Emitter 0 (Base)
EMITTER_BASE_REGISTER = 2


# ============================================================
# 5. Emitter 부품 비트값
# ============================================================

# Bit 9
GREEN_PRODUCT_BASE = 256

# Bit 12
GREEN_PRODUCT_LID = 2048

# 팔레트나 박스 없이 부품만 생성
NO_EMITTER_BASE = 0


# ============================================================
# 6. 로봇 이동 방향 설정
#
# 실제 움직임이 반대라면 True와 False를 서로 교환
# ============================================================

X_ENTRY_POSITION = False
X_EXIT_POSITION = True

Z_UP_POSITION = False
Z_DOWN_POSITION = True


# ============================================================
# 7. 버튼 접점 설정
#
# True:
#   정상 상태에서 Input=True
#   버튼을 누르면 Input=False
# ============================================================

STOP_ACTIVE_LOW = True
EMERGENCY_ACTIVE_LOW = True


# 입력 상태를 주기적으로 확인하려면 True
DEBUG_INPUTS = False


# Emitter 출력 펄스 시간
EMITTER_PULSE_SECONDS = 0.25


# ============================================================
# 8. 사용자 정의 예외
# ============================================================

class CycleAbort(Exception):
    """자동 운전 도중 정지 조건이 발생했을 때 사용"""
    pass


# ============================================================
# 9. PyModbus 버전 호환 요청 함수
# ============================================================

def modbus_request(method, *args, **kwargs):
    """
    PyModbus 버전에 따라 Slave ID 인자 이름이 다를 수 있다.

    최신 버전: device_id
    일부 버전: slave
    구버전: unit
    """

    try:
        return method(
            *args,
            **kwargs,
            device_id=SLAVE_ID
        )
    except TypeError:
        pass

    try:
        return method(
            *args,
            **kwargs,
            slave=SLAVE_ID
        )
    except TypeError:
        pass

    try:
        return method(
            *args,
            **kwargs,
            unit=SLAVE_ID
        )
    except TypeError:
        pass

    return method(*args, **kwargs)


# ============================================================
# 10. Modbus 입력 및 출력
# ============================================================

def read_inputs():
    """
    Discrete Input 0~10을 읽는다.
    """

    response = modbus_request(
        client.read_discrete_inputs,
        address=0,
        count=11
    )

    if response is None:
        raise ConnectionError(
            "Factory I/O 입력 응답이 없습니다."
        )

    if response.isError():
        raise ConnectionError(
            f"입력 읽기 실패: {response}"
        )

    if len(response.bits) < 11:
        raise ConnectionError(
            "읽은 입력 데이터 개수가 부족합니다."
        )

    return response.bits[:11]


def write_output(address, value):
    """
    Factory I/O Coil을 제어한다.
    """

    response = modbus_request(
        client.write_coil,
        address=address,
        value=bool(value)
    )

    if response is None:
        raise ConnectionError(
            f"Coil {address} 응답이 없습니다."
        )

    if response.isError():
        raise ConnectionError(
            f"Coil {address} 쓰기 실패: {response}"
        )


def write_holding_register(address, value):
    """
    Factory I/O Holding Register에 정수 값을 기록한다.
    """

    response = modbus_request(
        client.write_register,
        address=address,
        value=int(value)
    )

    if response is None:
        raise ConnectionError(
            f"Holding Register {address} 응답이 없습니다."
        )

    if response.isError():
        raise ConnectionError(
            f"Holding Register {address} 쓰기 실패: {response}"
        )


def write_counter(value):
    """
    Holding Register 0에 처리한 부품 수를 기록한다.
    """

    write_holding_register(
        COUNTER_REGISTER,
        value
    )


# ============================================================
# 11. 버튼 신호 처리
# ============================================================

def is_button_pressed(raw_value, active_low):
    """
    active_low=True인 NC 접점:

        raw=True  → 버튼 해제
        raw=False → 버튼 눌림

    active_low=False인 NO 접점:

        raw=False → 버튼 해제
        raw=True  → 버튼 눌림
    """

    if active_low:
        return not bool(raw_value)

    return bool(raw_value)


def get_stop_pressed(inputs):
    return is_button_pressed(
        inputs[STOP_BUTTON],
        STOP_ACTIVE_LOW
    )


def get_emergency_pressed(inputs):
    return is_button_pressed(
        inputs[EMERGENCY_STOP],
        EMERGENCY_ACTIVE_LOW
    )


# ============================================================
# 12. 표시등 및 안전 정지
# ============================================================

def set_lights(
    start=False,
    reset=False,
    stop=False
):
    write_output(START_LIGHT, start)
    write_output(RESET_LIGHT, reset)
    write_output(STOP_LIGHT, stop)


def stop_conveyors():
    write_output(ENTRY_CONVEYOR, False)
    write_output(EXIT_CONVEYOR, False)


def safe_stop():
    """
    Emitter와 컨베이어를 정지한다.

    운반 도중 Grab을 해제하면 물품이 떨어질 수 있으므로
    Grab 출력은 현재 상태를 유지한다.
    """

    write_output(EMITTER, False)
    write_output(ENTRY_CONVEYOR, False)
    write_output(EXIT_CONVEYOR, False)


# ============================================================
# 13. 자동 운전 중 정지 조건 확인
# ============================================================

def check_abort(inputs=None):
    if inputs is None:
        inputs = read_inputs()

    if get_emergency_pressed(inputs):
        raise CycleAbort(
            "비상정지가 눌렸습니다."
        )

    if get_stop_pressed(inputs):
        raise CycleAbort(
            "정지 버튼이 눌렸습니다."
        )

    if not inputs[AUTO_MODE]:
        raise CycleAbort(
            "자동 모드가 해제되었습니다."
        )

    if not inputs[FACTORY_RUNNING]:
        raise CycleAbort(
            "Factory I/O가 정지 또는 일시정지 상태입니다."
        )


# ============================================================
# 14. 특정 입력 신호 대기
# ============================================================

def wait_input(
    input_address,
    target=True,
    timeout=10
):
    """
    특정 센서가 목표 상태가 될 때까지 기다린다.
    """

    started_at = time.monotonic()

    while True:
        inputs = read_inputs()

        check_abort(inputs)

        current_value = bool(
            inputs[input_address]
        )

        if current_value == bool(target):
            return

        if time.monotonic() - started_at >= timeout:
            raise TimeoutError(
                f"Input {input_address} 신호 대기 시간 초과"
            )

        time.sleep(0.03)


# ============================================================
# 15. X축 및 Z축 이동
# ============================================================

def move_axis(
    output_address,
    moving_input_address,
    target_position,
    axis_name,
    start_timeout=2,
    finish_timeout=6
):
    """
    축 이동 순서:

    1. Move X 또는 Move Z 출력
    2. Moving X/Z가 ON 되는지 확인
    3. Moving X/Z가 OFF 되는지 확인
    """

    write_output(
        output_address,
        target_position
    )

    # 이동 시작 확인
    start_time = time.monotonic()

    while True:
        inputs = read_inputs()
        check_abort(inputs)

        if inputs[moving_input_address]:
            break

        if time.monotonic() - start_time >= start_timeout:
            raise TimeoutError(
                f"{axis_name}: 이동 시작 신호가 없습니다."
            )

        time.sleep(0.03)

    # 이동 완료 확인
    finish_time = time.monotonic()

    while True:
        inputs = read_inputs()
        check_abort(inputs)

        if not inputs[moving_input_address]:
            return

        if time.monotonic() - finish_time >= finish_timeout:
            raise TimeoutError(
                f"{axis_name}: 이동 완료 시간 초과"
            )

        time.sleep(0.03)


# ============================================================
# 16. Reset 전용 축 이동
#
# 이미 목표 위치인 경우 Moving 신호가 발생하지 않을 수 있다.
# 0.5초 동안 Moving 신호가 없으면 이미 도착한 것으로 판단한다.
# ============================================================

def move_axis_for_reset(
    output_address,
    moving_input_address,
    target_position,
    axis_name,
    timeout=6
):
    write_output(
        output_address,
        target_position
    )

    start_time = time.monotonic()
    moving_started = False

    while True:
        inputs = read_inputs()

        if get_emergency_pressed(inputs):
            raise CycleAbort(
                "초기화 중 비상정지가 발생했습니다."
            )

        moving = bool(
            inputs[moving_input_address]
        )

        if moving:
            moving_started = True

        if moving_started and not moving:
            print(f"{axis_name} 완료")
            return

        if (
            not moving_started
            and time.monotonic() - start_time >= 0.5
        ):
            print(
                f"{axis_name}: 이미 목표 위치로 판단"
            )
            return

        if time.monotonic() - start_time >= timeout:
            raise TimeoutError(
                f"{axis_name}: 초기화 시간 초과"
            )

        time.sleep(0.03)


# ============================================================
# 17. 시스템 초기화
# ============================================================

def reset_system():
    print()
    print("시스템 초기화 시작")

    safe_stop()

    set_lights(
        start=False,
        reset=True,
        stop=False
    )

    # Emitter가 팔레트나 박스를 생성하지 않도록 설정
    write_holding_register(
        EMITTER_BASE_REGISTER,
        NO_EMITTER_BASE
    )

    # 다음에 생성할 기본 부품은 Green Product Base
    write_holding_register(
        EMITTER_PART_REGISTER,
        GREEN_PRODUCT_BASE
    )

    # 충돌 방지를 위해 먼저 Z축 상승
    move_axis_for_reset(
        output_address=MOVE_Z,
        moving_input_address=MOVING_Z,
        target_position=Z_UP_POSITION,
        axis_name="Z축 상승"
    )

    # X축을 Entry 위치로 복귀
    move_axis_for_reset(
        output_address=MOVE_X,
        moving_input_address=MOVING_X,
        target_position=X_ENTRY_POSITION,
        axis_name="X축 Entry 복귀"
    )

    # 안전한 위치에서 Grab 해제
    write_output(GRAB, False)

    write_counter(0)

    time.sleep(0.3)

    set_lights(
        start=False,
        reset=False,
        stop=True
    )

    print("시스템 초기화 완료")


# ============================================================
# 18. Emitter 부품 생성
# ============================================================

def emit_one_part(part_number):
    """
    홀수 번째:
        Green Product Base 생성

    짝수 번째:
        Green Product Lid 생성
    """

    # 생성할 때 Entry Conveyor는 정지
    write_output(ENTRY_CONVEYOR, False)
    write_output(EMITTER, False)

    # 팔레트나 박스가 함께 생성되지 않도록 설정
    write_holding_register(
        EMITTER_BASE_REGISTER,
        NO_EMITTER_BASE
    )

    if part_number % 2 == 1:
        part_value = GREEN_PRODUCT_BASE
        part_name = "Green Product Base"

    else:
        part_value = GREEN_PRODUCT_LID
        part_name = "Green Product Lid"

    print()
    print(
        f"[0] {part_number}번째 부품 선택: "
        f"{part_name}"
    )

    # 생성할 부품 종류 지정
    write_holding_register(
        EMITTER_PART_REGISTER,
        part_value
    )

    # Register 값이 Factory I/O에 반영될 시간
    time.sleep(0.15)

    # Emitter ON
    write_output(EMITTER, True)

    # 한 개만 생성되도록 짧게 펄스 출력
    time.sleep(EMITTER_PULSE_SECONDS)

    # Emitter OFF
    write_output(EMITTER, False)

    time.sleep(0.1)

    print(
        f"[0] {part_name} 생성 명령 완료"
    )


# ============================================================
# 19. Exit Conveyor 배출
# ============================================================

def run_exit_conveyor():
    """
    Base와 Lid가 결합된 후 Exit Conveyor를 작동한다.

    센서가 이미 ON인 경우:
        센서가 OFF 될 때까지 기다린다.

    센서가 OFF인 경우:
        센서 ON 후 OFF가 될 때까지 기다린다.
    """

    print("[10] Exit Conveyor 작동")

    write_output(EXIT_CONVEYOR, True)

    try:
        inputs = read_inputs()

        if inputs[ITEM_AT_EXIT]:
            print(
                "[10] 결합된 제품이 Exit Sensor 위에 있습니다."
            )

            wait_input(
                input_address=ITEM_AT_EXIT,
                target=False,
                timeout=15
            )

        else:
            wait_input(
                input_address=ITEM_AT_EXIT,
                target=True,
                timeout=15
            )

            print(
                "[10] 결합된 제품이 Exit Sensor에 도착"
            )

            wait_input(
                input_address=ITEM_AT_EXIT,
                target=False,
                timeout=15
            )

        # 제품이 Remover까지 이동할 시간
        time.sleep(0.8)

    finally:
        write_output(EXIT_CONVEYOR, False)

    print("[10] 결합 제품 배출 완료")


# ============================================================
# 20. Pick & Place 1회 사이클
# ============================================================

def pick_and_place_cycle(part_number):
    print()
    print("========================================")
    print(
        f"Pick & Place 사이클 시작: "
        f"{part_number}번째 부품"
    )
    print("========================================")


    # --------------------------------------------------------
    # STEP 0: 홀수 Base / 짝수 Lid 생성
    # --------------------------------------------------------

    emit_one_part(part_number)


    # --------------------------------------------------------
    # STEP 1: Entry Conveyor로 부품 공급
    # --------------------------------------------------------

    print("[1] Entry Conveyor 작동")

    write_output(ENTRY_CONVEYOR, True)

    wait_input(
        input_address=ITEM_AT_ENTRY,
        target=True,
        timeout=15
    )

    write_output(ENTRY_CONVEYOR, False)

    print("[1] 부품이 Pick 위치에 도착")


    # --------------------------------------------------------
    # STEP 2: Z축 하강
    # --------------------------------------------------------

    print("[2] Z축 하강")

    move_axis(
        output_address=MOVE_Z,
        moving_input_address=MOVING_Z,
        target_position=Z_DOWN_POSITION,
        axis_name="Z축 하강"
    )


    # --------------------------------------------------------
    # STEP 3: 부품 파지
    # --------------------------------------------------------

    print("[3] 부품 파지")

    write_output(GRAB, True)

    time.sleep(0.3)

    wait_input(
        input_address=ITEM_DETECTED,
        target=True,
        timeout=3
    )

    print("[3] 부품 감지 확인")


    # --------------------------------------------------------
    # STEP 4: Z축 상승
    # --------------------------------------------------------

    print("[4] Z축 상승")

    move_axis(
        output_address=MOVE_Z,
        moving_input_address=MOVING_Z,
        target_position=Z_UP_POSITION,
        axis_name="Z축 상승"
    )


    # --------------------------------------------------------
    # STEP 5: X축 Exit 위치 이동
    # --------------------------------------------------------

    print("[5] X축 Exit 위치 이동")

    move_axis(
        output_address=MOVE_X,
        moving_input_address=MOVING_X,
        target_position=X_EXIT_POSITION,
        axis_name="X축 Exit 이동"
    )


    # --------------------------------------------------------
    # STEP 6: Exit 위치로 Z축 하강
    #
    # 홀수 번째에는 Base를 내려놓고,
    # 짝수 번째에는 Lid를 같은 위치의 Base 위에 내려놓는다.
    # --------------------------------------------------------

    if part_number % 2 == 1:
        print(
            "[6] Green Product Base 배치 위치로 Z축 하강"
        )

    else:
        print(
            "[6] Green Product Lid 조립 위치로 Z축 하강"
        )

    move_axis(
        output_address=MOVE_Z,
        moving_input_address=MOVING_Z,
        target_position=Z_DOWN_POSITION,
        axis_name="Z축 배치 위치 하강"
    )


    # --------------------------------------------------------
    # STEP 7: 부품 내려놓기
    #
    # Grab OFF 직후에도 부품이 그리퍼 아래에 있으므로
    # Item detected=False는 기다리지 않는다.
    # --------------------------------------------------------

    print("[7] Grab 해제")

    write_output(GRAB, False)

    # Base 배치 또는 Lid 결합이 안정화될 시간
    time.sleep(0.6)

    if part_number % 2 == 1:
        print(
            "[7] Green Product Base 배치 완료"
        )

    else:
        print(
            "[7] Green Product Lid 결합 완료"
        )


    # --------------------------------------------------------
    # STEP 8: Z축 상승
    # --------------------------------------------------------

    print("[8] Z축 상승")

    move_axis(
        output_address=MOVE_Z,
        moving_input_address=MOVING_Z,
        target_position=Z_UP_POSITION,
        axis_name="Z축 복귀 상승"
    )

    print("[8] Z축 상승 완료")


    # --------------------------------------------------------
    # STEP 9: X축 Entry 위치 복귀
    # --------------------------------------------------------

    print("[9] X축 Entry 위치 복귀")

    move_axis(
        output_address=MOVE_X,
        moving_input_address=MOVING_X,
        target_position=X_ENTRY_POSITION,
        axis_name="X축 Entry 복귀"
    )

    print("[9] X축 복귀 완료")


    # --------------------------------------------------------
    # STEP 10: 짝수 번째에만 Exit Conveyor 작동
    # --------------------------------------------------------

    if part_number % 2 == 0:
        print(
            f"[10] {part_number}번째 부품까지 조립 완료"
        )

        print(
            "[10] Base + Lid 완성품을 배출합니다."
        )

        run_exit_conveyor()

    else:
        print(
            f"[10] {part_number}번째 부품은 Base입니다."
        )

        print(
            "[10] 다음 Lid가 올 때까지 "
            "Exit 위치에서 대기합니다."
        )

        write_output(EXIT_CONVEYOR, False)


    print("========================================")
    print("Pick & Place 사이클 완료")
    print("========================================")


# ============================================================
# 21. 입력 상태 디버깅
# ============================================================

def print_input_debug(inputs):
    print(
        " | ".join([
            f"Entry={int(inputs[ITEM_AT_ENTRY])}",
            f"Exit={int(inputs[ITEM_AT_EXIT])}",
            f"MovingX={int(inputs[MOVING_X])}",
            f"MovingZ={int(inputs[MOVING_Z])}",
            f"Detected={int(inputs[ITEM_DETECTED])}",
            f"Start={int(inputs[START_BUTTON])}",
            f"Reset={int(inputs[RESET_BUTTON])}",
            f"StopRaw={int(inputs[STOP_BUTTON])}",
            f"EStopRaw={int(inputs[EMERGENCY_STOP])}",
            f"Auto={int(inputs[AUTO_MODE])}",
            f"Running={int(inputs[FACTORY_RUNNING])}"
        ])
    )


# ============================================================
# 22. 메인 프로그램
# ============================================================

def main():
    if not client.connect():
        print("Factory I/O Modbus 서버 연결 실패")
        print(
            f"연결 주소: {SERVER_IP}:{SERVER_PORT}"
        )
        return

    print("Factory I/O 연결 성공")

    automatic_running = False

    # 처리한 전체 부품 수
    # 홀수: 다음 생성 부품은 Lid
    # 짝수: 다음 생성 부품은 Base
    part_count = 0

    previous_start = False
    previous_reset = False
    previous_stop = False
    previous_emergency = False
    previous_auto = None

    last_debug_time = 0

    try:
        # ----------------------------------------------------
        # 최초 출력 상태 초기화
        # ----------------------------------------------------

        safe_stop()

        write_output(GRAB, False)

        set_lights(
            start=False,
            reset=False,
            stop=True
        )

        write_counter(0)

        write_holding_register(
            EMITTER_BASE_REGISTER,
            NO_EMITTER_BASE
        )

        write_holding_register(
            EMITTER_PART_REGISTER,
            GREEN_PRODUCT_BASE
        )


        # ----------------------------------------------------
        # 메인 반복
        # ----------------------------------------------------

        while True:
            inputs = read_inputs()

            start_pressed = bool(
                inputs[START_BUTTON]
            )

            reset_pressed = bool(
                inputs[RESET_BUTTON]
            )

            stop_pressed = get_stop_pressed(
                inputs
            )

            emergency_pressed = get_emergency_pressed(
                inputs
            )

            auto_enabled = bool(
                inputs[AUTO_MODE]
            )

            factory_running = bool(
                inputs[FACTORY_RUNNING]
            )


            # ------------------------------------------------
            # 입력 상태 확인
            # ------------------------------------------------

            if DEBUG_INPUTS:
                current_time = time.monotonic()

                if current_time - last_debug_time >= 0.5:
                    print_input_debug(inputs)
                    last_debug_time = current_time


            # ------------------------------------------------
            # 비상정지
            # ------------------------------------------------

            if emergency_pressed:
                automatic_running = False

                safe_stop()

                set_lights(
                    start=False,
                    reset=False,
                    stop=True
                )

                if not previous_emergency:
                    print()
                    print("비상정지 상태")
                    print(
                        "Emergency Stop을 해제한 후 "
                        "Reset 버튼을 누르세요."
                    )

                previous_emergency = True
                previous_start = start_pressed
                previous_reset = reset_pressed
                previous_stop = stop_pressed

                time.sleep(0.1)
                continue

            if previous_emergency:
                print(
                    "비상정지가 해제되었습니다."
                )

            previous_emergency = False


            # ------------------------------------------------
            # Factory I/O 정지 또는 일시정지
            # ------------------------------------------------

            if not factory_running:
                automatic_running = False

                safe_stop()

                set_lights(
                    start=False,
                    reset=False,
                    stop=True
                )

                previous_start = start_pressed
                previous_reset = reset_pressed
                previous_stop = stop_pressed

                time.sleep(0.1)
                continue


            # ------------------------------------------------
            # Reset 버튼 상승 에지
            # ------------------------------------------------

            if reset_pressed and not previous_reset:
                automatic_running = False

                try:
                    reset_system()

                    part_count = 0
                    write_counter(part_count)

                except (
                    CycleAbort,
                    TimeoutError,
                    ConnectionError
                ) as error:
                    print(
                        f"초기화 오류: {error}"
                    )

                    safe_stop()

                    set_lights(
                        start=False,
                        reset=False,
                        stop=True
                    )

            previous_reset = reset_pressed


            # ------------------------------------------------
            # Stop 버튼
            # ------------------------------------------------

            if stop_pressed:
                automatic_running = False

                safe_stop()

                set_lights(
                    start=False,
                    reset=False,
                    stop=True
                )

                if not previous_stop:
                    print(
                        "정지 버튼이 눌렸습니다."
                    )

                previous_stop = True
                previous_start = start_pressed

                time.sleep(0.05)
                continue

            if previous_stop:
                print(
                    "정지 버튼이 해제되었습니다."
                )

            previous_stop = False


            # ------------------------------------------------
            # Auto 모드
            # ------------------------------------------------

            if not auto_enabled:
                automatic_running = False

                safe_stop()

                set_lights(
                    start=False,
                    reset=False,
                    stop=True
                )

                if previous_auto is not False:
                    print(
                        "수동 모드 상태입니다."
                    )

                previous_auto = False
                previous_start = start_pressed

                time.sleep(0.05)
                continue

            if previous_auto is False:
                print(
                    "자동 모드가 선택되었습니다."
                )

            previous_auto = True


            # ------------------------------------------------
            # Start 버튼 상승 에지
            # ------------------------------------------------

            if start_pressed and not previous_start:
                automatic_running = True

                set_lights(
                    start=True,
                    reset=False,
                    stop=False
                )

                print()
                print("자동 운전 시작")

            previous_start = start_pressed


            # ------------------------------------------------
            # 자동 Pick & Place 사이클
            # ------------------------------------------------

            if automatic_running:
                try:
                    # 최초에는 1이므로 Green Product Base
                    next_part_number = part_count + 1

                    pick_and_place_cycle(
                        part_number=next_part_number
                    )

                    # 사이클이 완전히 성공한 뒤 카운트 증가
                    part_count = next_part_number

                    write_counter(part_count)

                    print(
                        f"현재 처리 부품 수: "
                        f"{part_count}"
                    )

                    print(
                        f"현재 완성 제품 수: "
                        f"{part_count // 2}"
                    )

                    if part_count % 2 == 0:
                        print(
                            "다음 생성 부품: "
                            "Green Product Base"
                        )
                    else:
                        print(
                            "다음 생성 부품: "
                            "Green Product Lid"
                        )


                except CycleAbort as error:
                    automatic_running = False

                    safe_stop()

                    set_lights(
                        start=False,
                        reset=False,
                        stop=True
                    )

                    print(
                        f"자동 운전 중지: {error}"
                    )


                except TimeoutError as error:
                    automatic_running = False

                    safe_stop()

                    set_lights(
                        start=False,
                        reset=False,
                        stop=True
                    )

                    print(
                        f"센서/축 동작 오류: {error}"
                    )


                except ConnectionError as error:
                    automatic_running = False

                    safe_stop()

                    set_lights(
                        start=False,
                        reset=False,
                        stop=True
                    )

                    print(
                        f"Modbus 통신 오류: {error}"
                    )

            time.sleep(0.05)


    except KeyboardInterrupt:
        print()
        print(
            "사용자가 프로그램을 종료했습니다."
        )


    except ConnectionError as error:
        print()
        print(
            f"Modbus 연결 오류: {error}"
        )


    finally:
        try:
            safe_stop()

            set_lights(
                start=False,
                reset=False,
                stop=True
            )

        except Exception:
            pass

        client.close()

        print(
            "Factory I/O 연결 종료"
        )


# ============================================================
# 23. 프로그램 실행
# ============================================================

if __name__ == "__main__":
    main()

Factory I/O 연결 성공
수동 모드 상태입니다.
자동 모드가 선택되었습니다.

자동 운전 시작

Pick & Place 사이클 시작: 1번째 부품

[0] 1번째 부품 선택: Green Product Base
[0] Green Product Base 생성 명령 완료
[1] Entry Conveyor 작동
[1] 부품이 Pick 위치에 도착
[2] Z축 하강
[3] 부품 파지
[3] 부품 감지 확인
[4] Z축 상승
[5] X축 Exit 위치 이동
[6] Green Product Base 배치 위치로 Z축 하강
[7] Grab 해제
[7] Green Product Base 배치 완료
[8] Z축 상승
[8] Z축 상승 완료
[9] X축 Entry 위치 복귀
[9] X축 복귀 완료
[10] 1번째 부품은 Base입니다.
[10] 다음 Lid가 올 때까지 Exit 위치에서 대기합니다.
Pick & Place 사이클 완료
현재 처리 부품 수: 1
현재 완성 제품 수: 0
다음 생성 부품: Green Product Lid

Pick & Place 사이클 시작: 2번째 부품

[0] 2번째 부품 선택: Green Product Lid
[0] Green Product Lid 생성 명령 완료
[1] Entry Conveyor 작동
[1] 부품이 Pick 위치에 도착
[2] Z축 하강
[3] 부품 파지
[3] 부품 감지 확인
[4] Z축 상승
[5] X축 Exit 위치 이동
[6] Green Product Lid 조립 위치로 Z축 하강
[7] Grab 해제
[7] Green Product Lid 결합 완료
[8] Z축 상승
[8] Z축 상승 완료
[9] X축 Entry 위치 복귀
[9] X축 복귀 완료
[10] 2번째 부품까지 조립 완료
[10] Base + Lid 완성품을 배출합니다.
[10] Exit Conveyor 작동
[10] 결합된 제품이 Exit Sensor 위에 있습니다.
[10] 결합 제품 배출 완료
Pick & Place 사이클 완료
현재 처리 부품 수